# Module 3.2 & 3.3 — Embedding Models and Selection

## OpenAI Embedding Models
| Model | Dimensions | Price | Best For |
|---|---|---|---|
| text-embedding-3-small | 1536 | $0.02/1M tokens | Best value |
| text-embedding-3-large | 3072 | $0.13/1M tokens | Highest quality |

## Open-Source (Ollama)
- `nomic-embed-text` — 768 dims, free, local
- `mxbai-embed-large` — 1024 dims
- `Gemma` via `langchain-ollama`

## MTEB Leaderboard
Visit https://huggingface.co/spaces/mteb/leaderboard to compare models on:
retrieval, clustering, classification, and more.

In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np, time

# ── Helper ────────────────────────────────────────────────────────────────────
def evaluate_model(embeddings_obj, name, queries, docs):
    """Embed queries and docs; report cosine similarities and timing."""
    t0   = time.time()
    q_embs = embeddings_obj.embed_documents(queries)
    d_embs = embeddings_obj.embed_documents(docs)
    elapsed = time.time() - t0

    def cos(a, b):
        a, b = np.array(a), np.array(b)
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    print(f"\n{'═'*55}")
    print(f"  Model   : {name}")
    print(f"  Dims    : {len(q_embs[0])}")
    print(f"  Time    : {elapsed:.2f}s for {len(queries)+len(docs)} texts")
    for q, qe in zip(queries, q_embs):
        sims = [(cos(qe, de), d) for de, d in zip(d_embs, docs)]
        top  = max(sims, key=lambda x: x[0])
        print(f"  Q: '{q[:40]}' → best match sim={top[0]:.4f}")
        print(f"     '{top[1][:60]}'")


queries = [
    "What is the capital of France?",
    "How does photosynthesis work?",
]
corpus = [
    "Paris is the capital and most populous city of France.",
    "London is the capital city of England and the United Kingdom.",
    "Photosynthesis is a process used by plants to convert light into energy.",
    "Respiration is the process of breaking down glucose to release energy.",
]

# small
evaluate_model(OpenAIEmbeddings(model="text-embedding-3-small"),
               "text-embedding-3-small", queries, corpus)

# large
evaluate_model(OpenAIEmbeddings(model="text-embedding-3-large"),
               "text-embedding-3-large", queries, corpus)


In [ ]:
# ── Ollama (local, free) ─────────────────────────────────────────────────────
# Requires Ollama running locally: https://ollama.com/download
# !ollama pull nomic-embed-text
# !pip install langchain-ollama

# from langchain_ollama import OllamaEmbeddings
# evaluate_model(OllamaEmbeddings(model="nomic-embed-text"), "nomic-embed-text", queries, corpus)
print("Uncomment the lines above once Ollama is running locally.")

# ── Dimension trade-offs ──────────────────────────────────────────────────────
print("\n📐 Dimension Trade-offs")
print(f"{'Dims':>6} | {'Storage per vector':>20} | {'Notes'}")
print("-"*55)
for dims in [384, 768, 1536, 3072]:
    storage_kb = (dims * 4) / 1024   # float32
    print(f"{dims:>6} | {storage_kb:>18.2f} KB | {'← OpenAI small' if dims==1536 else '← OpenAI large' if dims==3072 else ''}")
